This is an important part, I used an 4 gb dedicated video card (_NVIDIA GTX 1650_) and Iam using ollama with langchain to make the inferences on this course, I have not test the actual model (_batiai/gemma4-e2b:q4_) using tools but at east this model can fit on the VRAM and for now is working.

In [1]:
## loading lanchain_ollama for inference. It can also used requests
from langchain_ollama import ChatOllama

In [2]:
## Load the model. Ollama should be running (use `ollama serve` in terminal)
## and working in this URL http://localhost:11434/
retrieve_llm = ChatOllama(model="batiia-gemma-mini:latest")

In [3]:
prompt = "Where is located Guadalajara?"
response = retrieve_llm.invoke(prompt) ## this is how you can get the response from the model

In [6]:
## the response is a langchain object
response

AIMessage(content='Guadalajara is a major city located in **Jalisco**, Mexico.\n\nIt is situated in Western Mexico.', additional_kwargs={}, response_metadata={'model': 'batiia-gemma-mini:latest', 'created_at': '2026-06-16T06:19:29.863512713Z', 'done': True, 'done_reason': 'stop', 'total_duration': 131585152595, 'load_duration': 125393937087, 'prompt_eval_count': 29, 'prompt_eval_duration': 319235060, 'eval_count': 194, 'eval_duration': 5392308298, 'logprobs': None, 'model_name': 'batiia-gemma-mini:latest', 'model_provider': 'ollama'}, id='lc_run--019ecf13-bea0-7b71-a5fd-2299de09e7b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 194, 'total_tokens': 223})

In [7]:
## to see the text of the llm call
response.content

'Guadalajara is a major city located in **Jalisco**, Mexico.\n\nIt is situated in Western Mexico.'

In [8]:
## usage of the tokens models (does not matter when it runs on local)
response.usage_metadata

{'input_tokens': 29, 'output_tokens': 194, 'total_tokens': 223}

In [10]:
## in case to use another model, you can also calculate the price like this
## thinking this prices from gpt-5 mini
## Input: $0.75 per million tokens
## Output: $4.50 per million tokens

input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage_metadata['input_tokens'] * input_price +
    response.usage_metadata['output_tokens'] * output_price
)

cost

0.0008947499999999999

In [ ]:
## message history, is literally the conversation
## it can be list of dictionaries or list of langchain messages objects

INSTRUCTIONS = "You are a helpful assistant."
prompt = "Explain quantum computing in one sentence."

#method 1: list of dictionaries
message_history = [
    {"role": "system", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

# Pass it directly to the invoke method
response = retrieve_llm.invoke(message_history)
print(response.content)

Quantum computing is a revolutionary method of computation that harnesses the principles of quantum mechanics, such as superposition and entanglement, to process information in ways that allow it to solve complex problems far beyond the capability of classical computers.


In [13]:
#method 2: usage of langchain messages objects
from langchain_core.messages import SystemMessage, HumanMessage

message_history = [
    SystemMessage(content=INSTRUCTIONS),
    HumanMessage(content=prompt)
]

# Pass it directly to the invoke method
response = retrieve_llm.invoke(message_history)
print(response.content)

Quantum computing is a revolutionary method of computation that uses the principles of quantum mechanics, such as superposition and entanglement, to process information in ways that allow it to solve certain complex problems exponentially faster than classical computers.


In [15]:
## creating a function

def llm(instructions, user_prompt, model="batiia-gemma-mini:latest"):
    message_history = [
        SystemMessage(content=instructions),
        HumanMessage(content=user_prompt)
    ]

    # load the model
    retrieve_llm = ChatOllama(model=model)

    response = retrieve_llm.invoke(message_history)

    return response.content

In [16]:
## full rag function
from src.retrieve_documents import retrieve_documents
from src.searching_tool import fit_docs, search
from src.build_prompt import build_prompt, INSTRUCTIONS

documents = retrieve_documents()

index = fit_docs(documents)


def rag(query, model="batiia-gemma-mini:latest"):
    search_results = search(query, index, course="llm-zoomcamp")
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

Number of documents retrieved: 1342


In [17]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [18]:
rag("How do I get a certificate?")

'You can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to pass the Capstone project. Homework is not mandatory, but passing the Capstone project is required for the certificate.'